In [8]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


In [9]:
document1 = """
University Attendance Policy

Students are expected to attend all scheduled classes regularly. Attendance is important because it helps students understand lectures and complete their coursework. The university requires students to maintain at least 75 percent attendance in each subject. Students who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class. Students should check their attendance records regularly and report any mistakes to their teachers.

If a student is absent because of illness, the student should provide valid medical evidence to the department. The department may consider the medical reason, but medical evidence does not automatically remove the attendance shortage. In special cases, the department can review the student's situation and make a decision according to university rules.

Students should arrive on time and avoid unnecessary absences. Attendance records are maintained by the department throughout the semester.
"""

document2 = """
Assignment Submission Policy

Students must submit their assignments through the university learning portal before the given deadline. Every assignment has a specific submission date announced by the teacher. Students should upload their work in the required file format and check that the uploaded file is correct.

Late assignments may receive a penalty according to the course policy. Students are responsible for checking the submission status after uploading their work. If the portal shows an error, students should inform their teacher or the IT help desk.

If a student faces a serious medical emergency or a technical problem with the university portal, the teacher may allow an extension after reviewing the evidence. However, an extension is not automatic, and students must contact the teacher as soon as possible.

Teachers should clearly explain assignment requirements, deadlines, marking criteria, and accepted file formats. Students should avoid waiting until the last minute to upload their assignments.
"""

document3 = """
University IT Help Desk

The university IT help desk provides technical support to students and staff. Students can contact the help desk if they have problems with their university accounts, learning portal, email, or computer systems. The help desk normally operates from Monday to Saturday, between 9 AM and 5 PM.

Students should provide their registration number and a clear explanation of the problem when requesting support. For password problems, students can use the password recovery option on the login page. If the recovery option does not work, they should contact the IT help desk for further assistance.

The help desk usually responds to normal requests during working hours. However, urgent problems that affect online examinations or important university services may receive priority, even when the normal support schedule is busy.

Students should never share their passwords with other people. Staff members are responsible for protecting student account information and maintaining the university's computer systems.
"""

document4 = """
Updated University Attendance Policy

Students must maintain at least 80 percent attendance in each subject to be eligible for final examinations. Students with less than 80 percent attendance may not be allowed to appear in the final examination.

Teachers must record attendance in every class. Students should regularly check their attendance records and report any mistakes to their teachers.

Students with medical reasons must submit valid medical evidence to the department. The department may review special cases according to university rules.
"""

document5 = """
Assignment Extension Policy

Students must submit assignments before the announced deadline. An extension may only be granted when a student provides valid evidence of a serious medical emergency.

Technical problems with the university portal do not automatically qualify a student for an extension. Students must contact their teacher before the deadline if they face any submission problem.

The teacher is responsible for reviewing extension requests and informing students about the final decision.
"""

In [10]:
Documents = [document1, document2, document3, document4, document5]

In [11]:
def word_location(text):
    return [(m.start(),m.end()) for m in re.finditer(r"\S+",text)]


In [12]:
def create_chunks(text, size=40, overlap=0):
    position = word_location(text)
    chunks = []

    if overlap < 0 or overlap >= size:
        raise ValueError("Invalid overlap")
    step = size - overlap

    for i in range(0, len(position), step):
        start = position[i][0]
        end = position[min(i + size, len(position)) - 1][1]

        chunks.append({
            "id": len(chunks) + 1,
            "start": start,
            "end": end,
            "text": text[start:end]
        })
        if i + size >= len(position):
            break

    return chunks

In [13]:
import time

class Retriever:
    def __init__(self, records, backend="tfidf"):

        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")

        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]

        self.model_load_time = 0
        self.encoding_time = 0
        self.search_time = 0

        if backend == "tfidf":

            self.vectorizer = TfidfVectorizer()

            start_time = time.perf_counter()

            self.matrix = self.vectorizer.fit_transform(texts)

            self.encoding_time = time.perf_counter() - start_time

        elif backend == "embedding":

            from sentence_transformers import SentenceTransformer

            start_time = time.perf_counter()

            self.model = SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2", device="cpu")

            self.model_load_time = time.perf_counter() - start_time

            self.check_length(texts)

            start_time = time.perf_counter()

            self.matrix = self.model.encode(
                texts, normalize_embeddings=True)

            self.encoding_time = time.perf_counter() - start_time

        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):

        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)
                    ["input_ids"])

            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")

    def search(self, query, k=3, filters=None):

        start_time = time.perf_counter()

        if not query.strip() or k <= 0:
            return []

        if self.backend == "tfidf":

            q = self.vectorizer.transform([query])

            if q.nnz == 0:
                return []

            scores = (q @ self.matrix.T).toarray()[0]

        else:

            self.check_length([query])

            q = self.model.encode(
                [query], normalize_embeddings=True)[0]

            scores = self.matrix @ q

        allowed = [i for i, r in enumerate(self.records)
                   if all(r.get(key) == value
                          for key, value in (filters or {}).items())]

        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))

        results = [(self.records[i], float(scores[i]))
                   for i in ranked[:k]]

        self.search_time += time.perf_counter() - start_time

        return results

In [14]:
chunk_records = []

for i, document in enumerate(Documents, 1):
    for chunk in create_chunks(document, size=40, overlap=0):
        chunk["doc"] = f"document{i}"
        chunk["id"] = len(chunk_records) + 1
        chunk_records.append(chunk)

engine = Retriever(chunk_records, backend="tfidf")

In [17]:
print("Evidence assistant. Type exit to stop.")

while True:
    query = input("Question or Exit to stop: ")

    if query.strip().lower() == "exit":
        
        break

    hits = engine.search(query, k=3)

    if not hits:
        print("No candidate evidence; rephrase or contact a human.")
        continue

    print("Candidate evidence; relevance must be checked:")

    for number, (r, score) in enumerate(hits, 1):
        print(number, r["id"], f"{score:.3f}",
              "characters", r["start"], r["end"])

    chosen = input("Choose 1-3 to inspect; Enter if none apply: ")

    if not chosen.isdigit() or not 1 <= int(chosen) <= len(hits):
        print("No answer selected. Rephrase or contact a human.")
        continue

    r, score = hits[int(chosen)-1]

    print("Retrieved passage:", r["text"])
    print("Source:", r["doc"], "characters", r["start"], r["end"])

Evidence assistant. Type exit to stop.
Candidate evidence; relevance must be checked:
1 1 0.325 characters 1 298
2 3 0.275 characters 560 828
3 2 0.268 characters 299 559
Retrieved passage: University Attendance Policy

Students are expected to attend all scheduled classes regularly. Attendance is important because it helps students understand lectures and complete their coursework. The university requires students to maintain at least 75 percent attendance in each subject. Students
Source: document1 characters 1 298
Candidate evidence; relevance must be checked:
1 13 0.453 characters 1 263
2 2 0.320 characters 299 559
3 1 0.142 characters 1 298
Retrieved passage: Updated University Attendance Policy

Students must maintain at least 80 percent attendance in each subject to be eligible for final examinations. Students with less than 80 percent attendance may not be allowed to appear in the final examination.

Teachers must
Source: document4 characters 1 263
Candidate evidence; relevance